# ChainLens Phase 6 — Evaluation

> The fixture contains **one test-split labeled address**, `KAGGLE_FRAUD`. Its only label has `source='fixture'`.
> The Phase 6 evaluable-positive rule is `source != 'fixture'`, so the **real evaluable-positive count is 0**.
> The brief's locked `n=1` is therefore reported as the fixture-target count, never as a real-world positive.
> Precision, recall, F1, and PR-AUC are all reported as the string **"insufficient"**; no metric is computed on the fixture label.

This notebook is the evaluation report. It reports (a) data provenance, (b) the explicit positive-count disclosure,
(c) the ranked list and the fixture target's position, (d) the score distribution, (e) per-rule contributions,
(f) the `EXCHANGE_HOT` downweight arithmetic, and (g) an intentionally empty PR-curve placeholder. It makes **no performance claim**.


## 1. Data provenance

Counts are read directly from `labels`, `label_splits`, and `address_features`. The `as_of` dates are the snapshot dates carried through Phases 2–5.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

from chainlens.db import get_pg_engine
from chainlens.evaluation import (
    classification_metrics,
    evaluable_positives,
    ranked_list,
    rule_contribution_table,
)
from chainlens.fixtures import address_for

pd.set_option("display.max_columns", 50)
engine = get_pg_engine()

with engine.connect() as conn:
    labels = pd.read_sql(text("SELECT * FROM labels"), conn)
    splits = pd.read_sql(text("SELECT * FROM label_splits"), conn)
    features = pd.read_sql(text("SELECT address FROM address_features"), conn)
    scores = pd.read_sql(text("SELECT * FROM risk_scores"), conn)
    traces = pd.read_sql(text("SELECT address, rules_version, trace FROM score_trace"), conn)

provenance = (
    labels.groupby(["source", "as_of"], as_index=False)
    .size()
    .rename(columns={"size": "label_rows"})
    .sort_values(["source", "as_of"])
)
display(provenance)
print("label_splits:")
display(splits["split"].value_counts().rename_axis("split").to_frame("rows"))
print("feature-bearing labels by source/split:")
display(
    labels.merge(splits, on="address", how="inner")
    .merge(features, on="address", how="inner")
    .groupby(["source", "split"], as_index=False)
    .size()
    .rename(columns={"size": "rows"})
)


## 2. Evaluable-positive disclosure

`evaluable_positives` applies the locked formula: test-split addresses present in `address_features` with `source != 'fixture'`.
The fixture target is reported separately and is never passed into the metric function as a real positive.


In [ ]:
real_eval = evaluable_positives(labels, splits, features)
fixture_test = (
    labels.merge(splits, on="address", how="inner")
    .merge(features, on="address", how="inner")
    .query("split == 'test' and source == 'fixture'")
)

real_eval = real_eval.merge(scores[["address", "score"]], on="address", how="left")
y_true_real = (real_eval["label"].notna()).astype(int).to_numpy() if len(real_eval) else np.array([], dtype=int)
y_score_real = real_eval["score"].fillna(0).to_numpy(dtype=float) if len(real_eval) else np.array([], dtype=float)
metrics = classification_metrics(y_true_real, y_score_real)
f1_display = "insufficient" if metrics.get("insufficient") else f"{metrics['f1']:.3f}"

print(f"real evaluable positives (source != fixture): n={len(real_eval)}")
print(f"fixture test-split labeled addresses (not counted): n={len(fixture_test)}")
print(f"F1 = {f1_display!r}")
print(metrics)
if len(fixture_test):
    display(fixture_test[["address", "source", "label", "split"]])


## 3. Ranked list

The ranked list is sorted by current `score` descending. The top 20 are shown first; the fixture target is shown separately
because a score-0 fixture address is correctly ranked far below the top 20. `SANCTIONED_SEED` is also shown to make the
deck-pure policy visible: the seed is train-only and contributes no score by construction.


In [ ]:
full_ranked = ranked_list(scores, labels, splits, top_n=10_000)
full_ranked = full_ranked.reset_index(drop=True)
full_ranked["rank"] = full_ranked.index + 1

top20 = full_ranked.head(20)
display(top20)

fixture_target = full_ranked[full_ranked["address"] == address_for("KAGGLE_FRAUD")]
sanctioned_seed = full_ranked[full_ranked["address"] == address_for("SANCTIONED_SEED")]
print("Fixture target (test split; source='fixture'; NOT a real positive):")
display(fixture_target)
print("SANCTIONED_SEED (train split; deck-pure policy; score 0):")
display(sanctioned_seed)


## 4. Score distribution

The distribution is the fixture reality: `low=372`, `medium=5`, `high=0`. No fixture wallet fires three or more rules,
so the high tier is **untested on this fixture**. That is a data characteristic, not a defect in the rule engine.


In [ ]:
tiers = (
    scores.groupby("tier", as_index=False)
    .size()
    .rename(columns={"size": "wallets"})
    .sort_values("tier")
)
display(tiers)
ax = tiers.set_index("tier")["wallets"].plot.bar(color=["#9e9e9e", "#ffb300", "#d62728"][: len(tiers)])
ax.set_ylabel("wallets")
ax.set_title("Fixture score distribution by tier (high = 0; untested)")
plt.tight_layout()
plt.show()


## 5. Rule contribution table

Counts and sums come from the persisted `score_trace` rows for the current `rules_version`. R5 has zero hits on the fixture;
its mechanics are covered by the synthetic Apriori test, not by this ranking.


In [ ]:
current_version = scores["rules_version"].iloc[0]
current_traces = traces[traces["rules_version"] == current_version]
contributions = rule_contribution_table(current_traces)
display(contributions)
print(f"rules_version={current_version}; traces={len(current_traces)}")


## 6. `EXCHANGE_HOT` sanity check

`EXCHANGE_HOT` is a known exchange. Its raw score is 35, then the 0.3 exchange factor gives 10.5, which rounds to a final
score of 10 (banker's rounding). It ranks 25th. The trace must make this arithmetic reconstructable.


In [ ]:
exchange_address = address_for("EXCHANGE_HOT")
exchange = scores[scores["address"] == exchange_address].iloc[0]
exchange_trace = current_traces[current_traces["address"] == exchange_address].iloc[0]["trace"]
if isinstance(exchange_trace, str):
    import json
    exchange_trace = json.loads(exchange_trace)

print(f"address: {exchange_address}")
print(f"raw_score: {exchange['raw_score']}")
print(f"factor: {exchange_trace['downweight']['factor']}")
print(f"adjusted_raw_score: {exchange_trace['adjusted_raw_score']}")
print(f"final score: {exchange['score']}  tier: {exchange['tier']}")
print(f"rank: {int(full_ranked.loc[full_ranked['address'] == exchange_address, 'rank'].iloc[0])}")
print("arithmetic: 35 × 0.3 = 10.5 -> 10")
display(pd.DataFrame([
    {"rule": rule_id, "hit": rule["hit"], "weight": rule["weight"], "contribution": rule["contribution"]}
    for rule_id, rule in exchange_trace["rules"].items()
]))


## 7. PR-curve placeholder

The metric was considered and rejected. There are **0 real positives**, so a PR curve would be an empty-axis decoration.
The plot below is deliberately empty.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("PR curve not meaningful: n=0 real positives (n=1 fixture target)")
ax.text(0.5, 0.5, "insufficient positives; not reported", ha="center", va="center", fontsize=12)
plt.tight_layout()
plt.show()


## 8. Conclusion

The fixture validates the mechanics of the rules, not their predictive performance. The only test-split fixture label is
`KAGGLE_FRAUD`, and it is excluded from real-positive metrics by the locked `source != 'fixture'` rule. The ablation notebook
(`04_baseline_and_ablation.ipynb`) supplies the detection evidence: each rule is isolated, and removing it changes exactly its
own contribution. Real-data validation remains deferred.
